# D2C E-commerce Analysis — charts

Reads the query result CSVs in `results/` and builds the charts used in the README.
Data is from the store's Shopify exports (~64k orders, Jun–Nov 2026). All monetary
values are scaled by a constant factor to keep the repo public-safe — ratios and
trends are unaffected.

Run top to bottom; charts are written to `charts/`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
import os

os.makedirs('charts', exist_ok=True)
plt.rcParams.update({'font.size':11,'figure.facecolor':'white','axes.facecolor':'white',
    'axes.edgecolor':'#d5d5dd','axes.linewidth':.8})
INK='#1b1b2b'; MUTE='#8a8a9a'
C1='#2d6a6a'; C2='#3a7ca5'; C3='#d98c5f'

def rupee(x,_):
    if x>=1e7: return f'₹{x/1e7:.1f}Cr'
    if x>=1e5: return f'₹{x/1e5:.1f}L'
    if x>=1e3: return f'₹{x/1e3:.0f}k'
    return f'₹{x:.0f}'

## 1. Cohort retention by signup month

In [ ]:
co = pd.read_csv('results/cohort.csv')
piv = co.pivot(index='cohort_month', columns='mo', values='active')
base = piv[0]
ret = piv.div(base, axis=0) * 100
ret = ret.loc[:, [c for c in ret.columns if c <= 5]]
labels = [pd.to_datetime(m).strftime('%b %Y') for m in ret.index]

fig, ax = plt.subplots(figsize=(8,4.6))
cmap = mcolors.LinearSegmentedColormap.from_list('t', ['#f4f7f7', '#2d6a6a'])
data = ret.values.astype(float)
ax.imshow(np.where(np.isnan(data), np.nan, data), cmap=cmap, aspect='auto', vmin=0, vmax=15)
ax.set_xticks(range(ret.shape[1])); ax.set_xticklabels([f'M{c}' for c in ret.columns])
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        v = data[i,j]
        if not np.isnan(v):
            txt = f'{v:.0f}%' if j>0 else f'{v:.0f}%\n(n={int(base.iloc[i])})'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='white' if v>7 else INK)
ax.set_title('Cohort retention by signup month', fontweight='bold', fontsize=13, pad=12, color=INK)
ax.set_xlabel('Months since first order', fontsize=9, color=MUTE)
plt.tight_layout(); plt.savefig('charts/1_cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Average order value by channel

In [ ]:
a = pd.read_csv('results/aov_channel.csv')
a = a[a['orders']>=50].sort_values('aov')
fig, ax = plt.subplots(figsize=(8,4.2))
bars = ax.barh(a['channel'], a['aov'], color=C2, height=.62)
for b,v,o in zip(bars, a['aov'], a['orders']):
    ax.text(v+8, b.get_y()+b.get_height()/2, f'₹{v:,.0f}  ({o:,} orders)', va='center', fontsize=9, color=INK)
ax.set_xlim(0, a['aov'].max()*1.32); ax.set_xticks([])
ax.set_title('Average order value by sales channel', fontweight='bold', fontsize=13, pad=12, color=INK)
for s in ['top','right','bottom']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('charts/2_aov_channel.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Traffic by device type

In [ ]:
d = pd.read_csv('results/device.csv')
d = d[d['sessions']>0]
tot = d['sessions'].sum()
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(d['device_type'], d['sessions'], color=[C1,C3,C2,MUTE][:len(d)], width=.6)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}k'))
for b,v in zip(bars, d['sessions']):
    ax.text(b.get_x()+b.get_width()/2, v+tot*0.01, f'{100*v/tot:.1f}%', ha='center', fontsize=10, fontweight='bold', color=INK)
ax.set_title('Traffic by device type', fontweight='bold', fontsize=13, pad=12, color=INK)
ax.set_ylabel('Sessions', fontsize=9, color=MUTE)
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('charts/3_device_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Contribution margin by SKU

In [ ]:
m = pd.read_csv('results/margin.csv').sort_values('contribution_margin')
short = {'Mohini Itr - Pack of 3':'Mohini P3','Mohini Itr - Pack of 1':'Mohini P1',
         'Mohini Itr - Buy 3 Get 1':'Mohini 3+1','Bad Boy - Buy 2 Get 1':'Bad Boy 2+1',
         'Bad Boy - Pack of 1':'Bad Boy P1','Bad Boy - Buy 3 Get 1':'Bad Boy 3+1'}
m['lbl'] = m['product_name'].map(short).fillna(m['sku'])
fig, ax = plt.subplots(figsize=(8,4.6))
bars = ax.barh(m['lbl'], m['contribution_margin'], color=C1, height=.64)
for b,cm,mp in zip(bars, m['contribution_margin'], m['margin_pct']):
    ax.text(cm+cm*0.01, b.get_y()+b.get_height()/2, f'{rupee(cm,0)}  ·  {mp:.0f}% margin', va='center', fontsize=9, color=INK)
ax.set_xlim(0, m['contribution_margin'].max()*1.25)
ax.xaxis.set_major_formatter(FuncFormatter(rupee))
ax.set_title('Contribution margin by SKU', fontweight='bold', fontsize=13, pad=12, color=INK)
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('charts/4_margin_sku.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Monthly revenue trend

In [ ]:
dl = pd.read_csv('results/daily.csv'); dl['date'] = pd.to_datetime(dl['date'])
dl['month'] = dl['date'].dt.to_period('M').dt.to_timestamp()
mon = dl.groupby('month').agg(revenue=('revenue','sum'), orders=('orders','sum')).reset_index()
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(mon['month'], mon['revenue'], marker='o', color=C1, lw=2.2, markersize=6)
ax.fill_between(mon['month'], mon['revenue'], alpha=.08, color=C1)
ax.yaxis.set_major_formatter(FuncFormatter(rupee))
for x,y in zip(mon['month'], mon['revenue']):
    ax.annotate(rupee(y,0), (x,y), textcoords='offset points', xytext=(0,9), ha='center', fontsize=8, color=INK)
ax.set_title('Monthly revenue', fontweight='bold', fontsize=13, pad=12, color=INK)
ax.set_xticks(mon['month']); ax.set_xticklabels([m.strftime('%b') for m in mon['month']])
ax.set_ylim(0, mon['revenue'].max()*1.18)
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('charts/5_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()